In [ ]:
import pandas as pd
import sys

sys.path.append("/home/vlitoux/Documents/SLR-RAG/src")
from slrrag.settings import (
    PATH_CSV_01_QUERY_RESULTS,
    PANDERA_LAZY_VALIDATION,
)
from slrrag.settings import SchemaPapers, PATH_OUTPUT
from slrrag.toolbox.toolbox_notebooks import (
    plot_histogram,
    group_by_col_ref,
    explode_and_filter_on_list,
    explode_and_groupby_topic,
    filter_on_list,
)

df_papers = pd.read_csv(PATH_CSV_01_QUERY_RESULTS)
df_papers = SchemaPapers.validate(df_papers, lazy=PANDERA_LAZY_VALIDATION)
df_papers

In [ ]:
minimum_year_conference_paper = 2023
df_recent = df_papers[
    (
        (df_papers["type"] == "article")
        | (
            (df_papers["type"] == "conference-paper")
            & (df_papers["year"] >= minimum_year_conference_paper)
        )
    )
]
df_recent = df_recent[
    ~(df_recent["doi"].duplicated(keep="first")) | (df_recent["doi"].isna())
]  # TODO : BUG That should not be here
df_recent

# Topic analysis

## OpenAlex

#### Domain

In [ ]:
group_by_col_ref(df_recent, "domain_openalex")

In [ ]:
black_list_domain_openalex = {
    "Health Sciences",
    "Life Sciences",
}
df_openalex_filtering_domain = filter_on_list(
    df_recent, black_list_domain_openalex, "fields_openalex", blacklist=True
)
group_by_col_ref(df_openalex_filtering_domain, "domain_openalex")

### Fields

In [ ]:
group_by_col_ref(df_openalex_filtering_domain, "fields_openalex")

In [ ]:
black_list_fields_openalex = {
    "Agricultural and Biological Sciences",
    "Arts and Humanities",
    "Biochemistry, Genetics and Molecular Biology",
    "Chemical Engineering",
    "Chemistry",
    "Dentistry",
    "Health Professions",
    "Immunology and Microbiology",
    "Medicine",
    "Neuroscience",
    "Nursing",
    "Pharmacology, Toxicology and Pharmaceutics",
    "Psychology",
    "Business, Management and Accounting",
    "Materials Science",
    "Economics, Econometrics and Finance",
    "Physics and Astronomy",
}
df_openalex_filtering_fields = filter_on_list(
    df_recent, black_list_fields_openalex, "fields_openalex", blacklist=True
)
group_by_col_ref(df_openalex_filtering_fields, "fields_openalex")

### Subfields

In [ ]:
group_by_col_ref(df_openalex_filtering_fields, col_name="subfields_openalex")

In [ ]:
blacklist_subfields_openalex = {"Biomedical Engineering", "Epidemiology", "Energy"}
df_openalex_filtering_subfields = filter_on_list(
    df_openalex_filtering_fields,
    blacklist_subfields_openalex,
    "subfields_openalex",
    blacklist=True,
)
group_by_col_ref(df_openalex_filtering_subfields, "subfields_openalex")

In [ ]:
subfields = "Environmental Engineering"
df_openalex_filtering_subfields[
    df_openalex_filtering_subfields["subfields_openalex"] == subfields
].sort_values("relevance_openalex", ascending=False)[["title", "relevance_openalex"]]

In [ ]:
df_openalex_filtering_subfields.query("is_reference")

#### Topic

In [ ]:
group_by_col_ref(df_openalex_filtering_subfields, "topic_openalex")

In [ ]:
black_list_openalex_topic = {
    "Concrete Corrosion and Durability",
    "Arctic and Antarctic ice dynamics",
    "Fatigue and fracture mechanics",
    "Marine animal studies overview",
    "Spacecraft and Cryogenic Technologies",
    "Oceanographic and Atmospheric Processes",
    "Drilling and Well Engineering",
    "Fluid Dynamics and Vibration Analysis",
    "Engineering Structural Analysis Methods",
}
# Note : we skip lens filtering
df_openalex_filtering_topic = filter_on_list(
    df_openalex_filtering_subfields,
    blacklist_subfields_openalex,
    "topic_openalex",
    blacklist=True,
)
group_by_col_ref(df_openalex_filtering_topic, "topic_openalex")

## Topic in dimensions

In [ ]:
explode_and_groupby_topic(
    df_openalex_filtering_topic,
    name_topic_col="topic_dimensions",
    separator=";",
)

In [ ]:
black_list_topic_dimensions = {
    "A01 Clinical Medicine",
    "A01 Clinical Medicine; A03 Allied Health Professions, Dentistry, Nursing and Pharmacy",
    "A01 Clinical Medicine; A04 Psychology, Psychiatry and Neuroscience",
    "A02 Public Health, Health Services and Primary Care",
    "A02 Public Health, Health Services and Primary Care; A03 Allied Health Professions, Dentistry, Nursing and Pharmacy",
    "A03 Allied Health Professions, Dentistry, Nursing and Pharmacy",
    "A04 Psychology, Psychiatry and Neuroscience",
    "A05 Biological Sciences",
    "A06 Agriculture, Veterinary and Food Science",
    "C23 Education",
    "C24 Sport and Exercise Sciences, Leisure and Tourism",
    "D25 Area Studies",
    "D26 Modern Languages and Linguistics",
    "D28 History",
    "D32 Art and Design: History, Practice and Theory",
    "D33 Music, Drama, Dance, Performing Arts, Film and Screen Studies",
    "D34 Communication, Cultural and Media Studies, Library and Information Management",
}
df_dimensions_filtering_topic = explode_and_filter_on_list(
    df_openalex_filtering_topic,
    black_list_input=black_list_topic_dimensions,
    col_name="topic_dimensions",
    seaparator=";",
)
explode_and_groupby_topic(
    df_dimensions_filtering_topic, "topic_dimensions", separator=";"
)

## Topic in Lens

In [ ]:
explode_and_groupby_topic(
    df_dimensions_filtering_topic,
    name_topic_col="topic_lens",
    separator=";",
)

In [ ]:
black_list_lens_topic = {
    "Medicine",
    "Internal medicine",
    "Biology",
    "Chemistry",
    "Quantum Mechnic",
}
# Lens Topic are garbagely done.

# df_lens_filtering_topic = explode_and_filter_on_list(
#     df_lens_filtering_topic,
#     black_list_input=black_list_lens_topic,
#     col_name="topic_lens",
#     seaparator=";",
# )
# explode_and_groupby_topic(df_lens_filtering_topic,"topic_lens",separator=";")

# Keywords analysis

In [ ]:
explode_and_groupby_topic(
    df_dimensions_filtering_topic, name_topic_col="keywords_lens", separator=";"
).query("is_reference")

In [ ]:
plot_histogram(
    df_dimensions_filtering_topic,
    col_name="keywords_scopus",
    seperator="|",
    min_count_to_keep=15,
    height=500,
)

In [ ]:
from slrrag.upset_plotter import plot_and_save_upset

plot_and_save_upset(df_dimensions_filtering_topic, PATH_OUTPUT / "test_upset.html")

In [ ]:
from slrrag.settings import PATH_CSV_02_FILTERED_TOPIC


df_dimensions_filtering_topic.to_csv(PATH_CSV_02_FILTERED_TOPIC, index=False)